# 📊 Phân tích Dữ liệu Bán hàng
Công ty TNHH Phân phối Minh Long

**Học phần:** Xác suất thống kê và phân tích dữ liệu (FIT3004)
**Người thực hiện:** Trần Đức Huy — **Lớp:** D1026HNCNA1 — **MSSV:** DNU0666


In [ ]:
# Nhập các thư viện để xử lý dữ liệu và vẽ biểu đồ
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cài đặt hiển thị tiếng Việt nếu cần
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set(style='whitegrid')

# Định dạng trục tiền tệ sang dạng triệu VND (vd 10M) cho dễ đọc
fmt_trieu = plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M")

## 📥 Đọc dữ liệu

In [ ]:
from google.colab import files
uploaded = files.upload()
while not uploaded:
    print("Bạn chưa chọn file. Vui lòng upload sales_data.csv.")
    uploaded = files.upload()
ten_file = list(uploaded.keys())[0]
print(f"Đã upload: {ten_file}")

df = pd.read_csv(ten_file)
df['Ngay'] = pd.to_datetime(df['Ngay'])
df['Khu_vuc'] = df['Khu_vuc'].replace({'Ha Noi': 'Hà Nội', 'Da Nang': 'Đà Nẵng'})
df.head()

In [ ]:
# Kiểm tra tổng quan & làm sạch dữ liệu
print("Kích thước dữ liệu:", df.shape)
print("\nGiá trị thiếu:")
print(df.isnull().sum())
print("\nSố dòng trùng lặp:", df.duplicated().sum())

# Làm sạch: bỏ dòng thiếu và dòng trùng
df = df.dropna().drop_duplicates().reset_index(drop=True)
df.info()

## 🧩 Bước làm giàu dữ liệu (Enrich)
Từ cột `Ngay` truyền vào, tự động trích xuất thêm các trường thời gian:
- **Thang**: tháng (1, 2)
- **Thu**: thứ trong tuần (Thứ 2 → Chủ nhật)
- **Cuoi_tuan**: True nếu là Thứ 7 / Chủ nhật

Các cột mới này giúp phân tích doanh thu theo ngày trong tuần và so sánh giữa các tháng.


In [ ]:
# Trích xuất đặc trưng thời gian từ cột Ngay
ten_thu = ["Thứ 2", "Thứ 3", "Thứ 4", "Thứ 5", "Thứ 6", "Thứ 7", "Chủ nhật"]

df["Thang"] = df["Ngay"].dt.month
df["Thu"] = df["Ngay"].dt.dayofweek.map(dict(enumerate(ten_thu)))
df["Cuoi_tuan"] = df["Ngay"].dt.dayofweek >= 5   # 5=Thứ 7, 6=Chủ nhật

# Xem dữ liệu sau khi làm giàu
df.head()

## 📈 Thống kê mô tả theo khu vực

In [ ]:
# Tạo hàm thống kê
def summary_stats(area):

    sub_df = df[df['Khu_vuc'] == area]
    mean = sub_df['Doanh_thu'].mean()
    std = sub_df['Doanh_thu'].std()
    var = sub_df['Doanh_thu'].var()
    min_val = sub_df['Doanh_thu'].min()
    max_val = sub_df['Doanh_thu'].max()
    below_mean = (sub_df['Doanh_thu'] < mean).sum()

    print(f"--- {area} ---")
    print(f"Trung bình: {mean:.2f}")
    print(f"Độ lệch chuẩn: {std:.2f}")
    print(f"Phương sai: {var:.2f}")
    print(f"Min: {min_val}")
    print(f"Max: {max_val}")
    print(f"Số ngày doanh thu < trung bình: {below_mean}")
    print()

# Thực hiện với từng khu vực
for area in df['Khu_vuc'].unique():
    summary_stats(area)

## 📊 Trực quan hóa dữ liệu

In [ ]:
# Histogram - mỗi khu vực một biểu đồ riêng
khu_vuc_list = ['Hà Nội', 'TP.HCM', 'Đà Nẵng']
mau = {'Hà Nội': '#2E86AB', 'TP.HCM': '#E4572E', 'Đà Nẵng': '#3CB371'}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, kv in zip(axes, khu_vuc_list):
    du_lieu = df.loc[df['Khu_vuc'] == kv, 'Doanh_thu']
    sns.histplot(du_lieu, bins=12, kde=True, color=mau[kv], ax=ax)
    ax.axvline(du_lieu.mean(), color='black', ls='--', lw=1.2, label='Trung bình')
    ax.set_title(f"Phân bố doanh thu – {kv}")
    ax.set_xlabel("Doanh thu (VND)")
    ax.set_ylabel("Số ngày")
    ax.xaxis.set_major_formatter(fmt_trieu)
    ax.legend()
plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Khu_vuc', y='Doanh_thu')
plt.title("Biểu đồ hộp doanh thu theo khu vực")
plt.gca().yaxis.set_major_formatter(fmt_trieu)
plt.show()

# Line chart
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Ngay', y='Doanh_thu', hue='Khu_vuc', marker='o')
plt.title("Doanh thu theo thời gian")
plt.ylabel("Doanh thu (VND)")
plt.gca().yaxis.set_major_formatter(fmt_trieu)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 🔍 Mở rộng: Trung bình trượt 7 ngày

In [ ]:
# Áp dụng trung bình trượt (rolling mean) 7 ngày cho từng khu vực
# để làm mượt dao động ngày và đánh giá xu hướng doanh thu theo thời gian.

plt.figure(figsize=(12, 6))
for area in df['Khu_vuc'].unique():
    sub_df = df[df['Khu_vuc'] == area].sort_values('Ngay')
    rolling_mean = sub_df['Doanh_thu'].rolling(window=7, min_periods=1).mean()
    plt.plot(sub_df['Ngay'], rolling_mean, marker='o', label=area)

plt.title("Trung bình trượt 7 ngày theo khu vực")
plt.xlabel("Ngày")
plt.ylabel("Doanh thu (VND)")
plt.gca().yaxis.set_major_formatter(fmt_trieu)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 📝 So sánh và nhận xét

In [ ]:
# So sánh mức độ ổn định giữa các khu vực dựa trên độ lệch chuẩn
std_by_area = df.groupby('Khu_vuc')['Doanh_thu'].std().sort_values()
print("Độ lệch chuẩn theo khu vực (thấp = ổn định):")
print(std_by_area.round(0))

print(f"\n=> Ổn định nhất: {std_by_area.index[0]}")
print(f"=> Biến động cao nhất: {std_by_area.index[-1]}")

**Nhận xét:**
- **TP.HCM**: doanh thu trung bình cao nhất nhưng độ lệch chuẩn lớn nhất → tiềm năng nhưng cần kiểm soát rủi ro.
- **Đà Nẵng**: doanh thu thấp hơn nhưng ổn định nhất → phù hợp chiến lược giữ chân khách hàng dài hạn.
- **Hà Nội**: ở mức trung bình cả về doanh thu lẫn biến động.


## 📅 Phân tích nâng cao 1: Hiệu ứng ngày trong tuần
Sử dụng cột `Thu` và `Cuoi_tuan` (từ bước làm giàu dữ liệu) để xem doanh thu có
phụ thuộc vào ngày trong tuần hay không.


In [ ]:
# Doanh thu trung bình theo thứ trong tuần
tb_theo_thu = df.groupby("Thu")["Doanh_thu"].mean().reindex(ten_thu)

plt.figure(figsize=(10, 5))
sns.barplot(x=tb_theo_thu.index, y=tb_theo_thu.values,
            hue=tb_theo_thu.index, palette="viridis", legend=False)
plt.title("Doanh thu trung bình theo ngày trong tuần")
plt.xlabel("Thứ"); plt.ylabel("Doanh thu trung bình (VND)")
# Hiển thị trục Y dạng triệu VND (M) cho dễ đọc
plt.gca().yaxis.set_major_formatter(fmt_trieu)
plt.tight_layout(); plt.show()

# So sánh cuối tuần vs ngày thường
tb_ct = df.groupby("Cuoi_tuan")["Doanh_thu"].mean()
print(f"Ngày thường: {tb_ct[False]:,.0f} VND")
print(f"Cuối tuần  : {tb_ct[True]:,.0f} VND")
chenh = (tb_ct[True] - tb_ct[False]) / tb_ct[False] * 100
print(f"=> Cuối tuần {'cao hơn' if chenh > 0 else 'thấp hơn'} ngày thường {abs(chenh):.2f}%")

## 🗓️ Phân tích nâng cao 2: So sánh Tháng 1 vs Tháng 2
Sử dụng cột `Thang` để so sánh doanh thu trung bình mỗi khu vực giữa hai tháng.


In [ ]:
# Bảng doanh thu TB theo khu vực x tháng
bang_thang = df.pivot_table(index="Khu_vuc", columns="Thang",
                            values="Doanh_thu", aggfunc="mean").round(0)
bang_thang.columns = ["Tháng 1", "Tháng 2"]
bang_thang["% thay đổi"] = ((bang_thang["Tháng 2"] - bang_thang["Tháng 1"])
                            / bang_thang["Tháng 1"] * 100).round(2)
print(bang_thang)

# Biểu đồ cột nhóm
bang_thang[["Tháng 1", "Tháng 2"]].plot(kind="bar", figsize=(9, 5))
plt.title("So sánh doanh thu trung bình: Tháng 1 vs Tháng 2")
plt.xlabel("Khu vực"); plt.ylabel("Doanh thu trung bình (VND)")
plt.gca().yaxis.set_major_formatter(fmt_trieu)
plt.xticks(rotation=0); plt.legend(title="Tháng")
plt.tight_layout(); plt.show()